In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.olist_raw;
CREATE VOLUME IF NOT EXISTS workspace.olist_raw.landing;

In [0]:
dbutils.fs.ls("/Volumes/workspace/olist_raw/landing/")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.olist_bronze;
CREATE SCHEMA IF NOT EXISTS workspace.olist_silver;
CREATE SCHEMA IF NOT EXISTS workspace.olist_gold;

In [0]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import current_timestamp, col


In [0]:
def load_bronze_table(file_path, table_name, schema):
    print("=" * 80)
    print(f"Starting Bronze Load for: {table_name}")
    print(f"Source File: {file_path}")

    df = (
        spark.read
        .option("header", True)
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .schema(schema)
        .csv(file_path)
        .withColumn("source_file", col("_metadata.file_path"))
        .withColumn("ingestion_timestamp", current_timestamp())
    )

    total_count = df.count()
    corrupt_count = df.filter(col("_corrupt_record").isNotNull()).count()

    print(f"Total records read: {total_count}")
    print(f"Corrupt records found: {corrupt_count}")

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )

    print(f"Table written successfully: {table_name}")
    display(spark.table(table_name).limit(10))

    return df

In [0]:
# Customers schema
schema_customers = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_unique_id", StringType(), True),
    StructField("customer_zip_code_prefix", StringType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])
# Orders Schema
schema_orders = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", StringType(), True),
    StructField("order_approved_at", StringType(), True),
    StructField("order_delivered_carrier_date", StringType(), True),
    StructField("order_delivered_customer_date", StringType(), True),
    StructField("order_estimated_delivery_date", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])
# Order Items Schema
schema_order_items = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_item_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", StringType(), True),
    StructField("price", StringType(), True),
    StructField("freight_value", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])
# Order Payments Schema
schema_order_payments = StructType([
    StructField("order_id", StringType(), True),
    StructField("payment_sequential", StringType(), True),
    StructField("payment_type", StringType(), True),
    StructField("payment_installments", StringType(), True),
    StructField("payment_value", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])
# Order Reviews Schema
schema_order_reviews = StructType([
    StructField("review_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("review_score", StringType(), True),
    StructField("review_comment_title", StringType(), True),
    StructField("review_comment_message", StringType(), True),
    StructField("review_creation_date", StringType(), True),
    StructField("review_answer_timestamp", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])
# Products Schema
schema_products = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_category_name", StringType(), True),
    StructField("product_name_lenght", StringType(), True),
    StructField("product_description_lenght", StringType(), True),
    StructField("product_photos_qty", StringType(), True),
    StructField("product_weight_g", StringType(), True),
    StructField("product_length_cm", StringType(), True),
    StructField("product_height_cm", StringType(), True),
    StructField("product_width_cm", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])
#Sellers Schema
schema_sellers = StructType([
    StructField("seller_id", StringType(), True),
    StructField("seller_zip_code_prefix", StringType(), True),
    StructField("seller_city", StringType(), True),
    StructField("seller_state", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])
#Geolocation Schema
schema_geolocation = StructType([
    StructField("geolocation_zip_code_prefix", StringType(), True),
    StructField("geolocation_lat", StringType(), True),
    StructField("geolocation_lng", StringType(), True),
    StructField("geolocation_city", StringType(), True),
    StructField("geolocation_state", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])
# Product category translation schema
schema_product_category_name = StructType([
    StructField("product_category_name", StringType(), True),
    StructField("product_category_name_english", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])

In [0]:
df_customers = load_bronze_table(
    file_path="/Volumes/workspace/olist_raw/landing/olist_customers_dataset.csv",
    table_name="workspace.olist_bronze.customers",
    schema=schema_customers
)
df_orders = load_bronze_table(
    file_path="/Volumes/workspace/olist_raw/landing/olist_orders_dataset.csv",
    table_name="workspace.olist_bronze.orders",
    schema=schema_orders
)
df_order_items = load_bronze_table(
    file_path="/Volumes/workspace/olist_raw/landing/olist_order_items_dataset.csv",
    table_name="workspace.olist_bronze.order_items",
    schema=schema_order_items
)
df_order_payments = load_bronze_table(
    file_path="/Volumes/workspace/olist_raw/landing/olist_order_payments_dataset.csv",
    table_name="workspace.olist_bronze.order_payments",
    schema=schema_order_payments
)
df_order_reviews = load_bronze_table(
    file_path="/Volumes/workspace/olist_raw/landing/olist_order_reviews_dataset.csv",
    table_name="workspace.olist_bronze.order_reviews",
    schema=schema_order_reviews
)
df_products = load_bronze_table(
    file_path="/Volumes/workspace/olist_raw/landing/olist_products_dataset.csv",
    table_name="workspace.olist_bronze.products",
    schema=schema_products
)
df_sellers = load_bronze_table(
    file_path="/Volumes/workspace/olist_raw/landing/olist_sellers_dataset.csv",
    table_name="workspace.olist_bronze.sellers",
    schema=schema_sellers
)
df_geolocation = load_bronze_table(
    file_path="/Volumes/workspace/olist_raw/landing/olist_geolocation_dataset.csv",
    table_name="workspace.olist_bronze.geolocation",
    schema=schema_geolocation
)
df_product_category_name = load_bronze_table(
    file_path="/Volumes/workspace/olist_raw/landing/product_category_name_translation.csv",
    table_name="workspace.olist_bronze.product_category_name",
    schema=schema_product_category_name
)
